In [ ]:
import numpy as np
import random
import operator

# Set random seed for consistent, reproducible results
random.seed(42)
np.random.seed(42)

class City:
    def __init__(self, name, x, y):
        self.name = name
        self.x = x
        self.y = y

    def distance(self, city):
        return np.sqrt((self.x - city.x)**2 + (self.y - city.y)**2)

    def __repr__(self):
        return f"{self.name}({self.x},{self.y})"

class Fitness:
    def __init__(self, route):
        self.route = route
        self.distance = 0

    def routeDistance(self):
        if self.distance == 0:
            path_distance = 0
            for i in range(len(self.route)):
                path_distance += self.route[i].distance(self.route[(i + 1) % len(self.route)])
            self.distance = path_distance
        return self.distance

    def routeFitness(self):
        return 1 / float(self.routeDistance())

def create_route(cityList):
    return random.sample(cityList, len(cityList))

def initial_population(popSize, cityList):
    return [create_route(cityList) for _ in range(popSize)]

def rank_routes(population):
    fitnessResults = {}
    for i in range(len(population)):
        fitnessResults[i] = Fitness(population[i]).routeFitness()
    return sorted(fitnessResults.items(), key=operator.itemgetter(1), reverse=True)

def selection(popRanked, eliteSize):
    # FIXED: Extracts the integer index explicitly out of the (index, fitness) tuple
    selectionResults = [popRanked[i][0] for i in range(eliteSize)]
    for _ in range(len(popRanked) - eliteSize):
        aspirants = random.sample(popRanked, 3)
        best = max(aspirants, key=lambda x: x[1]) # Compare by fitness value
        selectionResults.append(best[0])
    return selectionResults

def mating_pool(population, selectionResults):
    return [population[idx] for idx in selectionResults]

def breed(parent1, parent2):
    geneA, geneB = int(random.random() * len(parent1)), int(random.random() * len(parent1))
    startGene, endGene = min(geneA, geneB), max(geneA, geneB)
    childP1 = parent1[startGene:endGene]
    childP2 = [item for item in parent2 if item not in childP1]
    return childP1 + childP2

def breed_population(pool, eliteSize):
    children = [pool[i] for i in range(eliteSize)]
    length = len(pool) - eliteSize
    pool_sample = random.sample(pool, len(pool))
    for i in range(length):
        child = breed(pool_sample[i], pool_sample[len(pool)-i-1])
        children.append(child)
    return children

def mutate(individual, mutationRate):
    for swapped in range(len(individual)):
        if random.random() < mutationRate:
            swapWith = int(random.random() * len(individual))
            individual[swapped], individual[swapWith] = individual[swapWith], individual[swapped]
    return individual

def mutate_population(population, mutationRate):
    return [mutate(ind.copy(), mutationRate) for ind in population]

def next_generation(currentGen, eliteSize, mutationRate):
    popRanked = rank_routes(currentGen)
    selectionResults = selection(popRanked, eliteSize)
    pool = mating_pool(currentGen, selectionResults)
    children = breed_population(pool, eliteSize)
    return mutate_population(children, mutationRate)

def genetic_algorithm(cityList, popSize, eliteSize, mutationRate, generations):
    pop = initial_population(popSize, cityList)
    initial_best_index = rank_routes(pop)[0][0]
    initial_distance = Fitness(pop[initial_best_index]).routeDistance()
    print(f"\n[+] Initial Random Route Distance: {initial_distance:.2f} km")

    log_interval = max(1, generations // 4)

    for generation in range(generations):
        pop = next_generation(pop, eliteSize, mutationRate)
        if (generation + 1) % log_interval == 0 or (generation + 1) == generations:
            current_best_idx = rank_routes(pop)[0][0]
            current_best = Fitness(pop[current_best_idx]).routeDistance()
            print(f" -> Distance at Iteration {generation + 1}: {current_best:.2f} km")

    final_ranked = rank_routes(pop)
    best_route = pop[final_ranked[0][0]]
    final_distance = Fitness(best_route).routeDistance()

    print(f"\n[🥇] Final Optimized Absolute Minimum Distance: {final_distance:.2f} km")
    print(f"Perfect Stop Sequence Loop:")
    for rank, stop in enumerate(best_route, 1):
        print(f"  {rank}. {stop}")

if __name__ == '__main__':
    print("=== Delivery Route Genetic Optimizer Configuration ===")

    delivery_stops = []

    try:
        num_stops = int(input("How many delivery stops (including Hub) do you want to add? "))
        if num_stops < 3:
            print("Error: You need at least 3 stops to optimize a path loop. Defaulting to 4.")
            num_stops = 4

        print("\nEnter details for each stop:")
        for i in range(num_stops):
            name = input(f"  Name for Stop {i+1} (e.g., Warehouse, Stop_A): ").strip()
            if not name:
                name = f"Stop_{i+1}"
            x = float(input(f"    Enter X coordinate for {name}: "))
            y = float(input(f"    Enter Y coordinate for {name}: "))
            delivery_stops.append(City(name, x, y))

        generations = int(input("\nHow many optimization iterations (generations) to run? (e.g., 1000): "))
        if generations <= 0:
            generations = 1000

    except ValueError:
        print("\nInvalid input. Loading your default parameters for Bangalore destinations...")
        delivery_stops = [
            City("BMSCE", 0, 0),
            City("KR MARKET", 100, 55),
            City("WHITEFIELD", -1000, 200),
            City("ELECTRONIC CITY", 100, 32),
            City("HOSUR", 2500, 5899)
        ]
        generations = 2000

    population_size = 100
    elite_size = 20
    mutation_rate = 0.01

    print("\nRunning algorithm... please wait.")
    genetic_algorithm(
        cityList=delivery_stops,
        popSize=population_size,
        eliteSize=elite_size,
        mutationRate=mutation_rate,
        generations=generations
    )
23

=== Delivery Route Genetic Optimizer Configuration ===
How many delivery stops (including Hub) do you want to add? 2
Error: You need at least 3 stops to optimize a path loop. Defaulting to 4.

Enter details for each stop:
  Name for Stop 1 (e.g., Warehouse, Stop_A): GHRYH
    Enter X coordinate for GHRYH: 1000
    Enter Y coordinate for GHRYH: 45
  Name for Stop 2 (e.g., Warehouse, Stop_A): 5TSGNY
    Enter X coordinate for 5TSGNY: 5279537
    Enter Y coordinate for 5TSGNY: 4598
  Name for Stop 3 (e.g., Warehouse, Stop_A): SRHEBSRHEB
    Enter X coordinate for SRHEBSRHEB: 49805
    Enter Y coordinate for SRHEBSRHEB: 4985\

Invalid input. Loading your default parameters for Bangalore destinations...

Running algorithm... please wait.

[+] Initial Random Route Distance: 14153.36 km
 -> Distance at Iteration 500: 14153.36 km
 -> Distance at Iteration 1000: 14153.36 km
 -> Distance at Iteration 1500: 14153.36 km
 -> Distance at Iteration 2000: 14153.36 km

[🥇] Final Optimized Absolute Mini